# Scan HV dei pannelli di scintillatore — metodo di analisi

## Setup sperimentale

Due pannelli di scintillatore **identici**, **EJ-200** (Eljen Technology),
**100 × 50 × 2.5 cm³** ($A = 0.5\ \mathrm{m^2}$),
montati uno sopra l'altro a distanza **$d = 125$ cm** (misurata), ciascuno letto da **un singolo PMT**.

Uno dei due pannelli fa da **trigger esterno** a HV fissa (1100 V); l'altro è il **pannello sotto
studio**, di cui si scansiona l'HV da 800 a 1700 V a passi di 50 V. Le due configurazioni sono
simmetriche e si distinguono dal token `TriggExt` nel nome del file:

| file | trigger | sotto studio |
|------|---------|--------------|
| `MV_PMT_study_BottomTriggExt1100V_Top<HV>V_...` | Bottom | **Top** |
| `MV_PMT_study_TopTriggExt1100V_Bottom<HV>V_...` | Top | **Bottom** |

Il PMT sotto studio è **sempre sul canale 0** del digitizer (si scambia il cavo, non il canale).

**DAQ**: 2.5 GS/s ($\Delta t = 0.4$ ns), 1024 campioni per traccia (≈ 410 ns), **1000 trigger per run**,
`TriggerMode = External`, soglia hardware del trigger **−15 mV** (esclude i dark count dal
trigger → purezza ≈ 1). L'header dei file `.h5` **non contiene** durata né timestamp.

---

## Catena di analisi

1. **Baseline** — sottrazione della media nella finestra `BASELINE_WINDOW_NS`.

2. **Modo comune** — il digitizer introduce un disturbo comune a tutti i canali:
   $$\mathrm{signal\_corr} = \mathrm{ch}_0 - \alpha \cdot \mathrm{median}(\mathrm{ch}_1 \dots \mathrm{ch}_7)$$
   con $\alpha$ stimato **per evento** ai minimi quadrati in una finestra senza segnale.
   Misurato: RMS del rumore $0.736 \to 0.571$ mV (guadagno ×1.29). La sottrazione secca
   $\mathrm{ch}_0 - \mathrm{ch}_1$ **non** funziona (×0.99): ch1 è più rumoroso di ch0, quindi
   il rumore indipendente che si aggiunge annulla il modo comune che si toglie.

3. **Gate temporale** — i muoni sono agganciati al trigger: $t_{\min} = 96\text{–}100$ ns con
   IQR ≈ 12 ns. Il gate `INTEGRATION_WINDOW_NS = (80, 120)` ns li contiene tutti.

4. **Soglia** in ampiezza (`NEGATIVE_THRESHOLD_MV`) valutata **solo dentro il gate**.

5. **Carica** — integrale trapezoidale sul gate. Una finestra di **piedistallo** della stessa
   larghezza (`PEDESTAL_WINDOW_NS`, 250–290 ns) viene integrata per **tutti** gli eventi: media $\approx 0.5$ mV·ns,
   $\sigma \approx 10.7$ mV·ns, contro una mediana di segnale di ~1570 mV·ns
   (**separazione ~147σ**). La media compatibile con zero conferma che baseline e modo comune
   non introducono bias.

6. **Selezione in carica** — vedi sotto.

---

## Relazione fondamentale: cosa misura davvero la frazione

Detta $G$ l'**accettanza geometrica** (probabilità che un muone che ha attraversato un pannello
attraversi anche l'altro), la frazione di trigger con segnale nel pannello sotto studio è

$$f(\mathrm{HV}) \;=\; \frac{n_{\text{segnale}}}{N_{\text{trigger}}}
\;=\; \frac{R_{\text{geom}} \cdot \varepsilon_{\text{trig}} \cdot \varepsilon_{\text{scan}}(\mathrm{HV})}
{R_{\text{single}} \cdot \varepsilon_{\text{trig}}}
\;=\; G \cdot \varepsilon_{\text{scan}}(\mathrm{HV})$$

Conseguenze importanti:

- **L'efficienza del pannello di trigger si semplifica.** Da questi dati **non** è estraibile:
  servirebbe un terzo rivelatore (telescopio a triplice coincidenza).
- La grandezza da confrontare fra le due configurazioni è la **frazione** $n/1000$, non la rate
  in Hz: le due configurazioni hanno trigger rate diversi, quindi in Hz il confronto è falsato.
- A piena efficienza $f \to G$.

**Accettanza calcolata** (Monte Carlo, flusso $\propto \cos^2\theta$, pannelli 100×50 allineati):

| $d$ [cm] | 50 | 75 | 100 | **125** | 150 | 200 |
|---|---|---|---|---|---|---|
| $G$ | 0.437 | 0.293 | 0.205 | **0.149** | 0.113 | 0.069 |

Il disallineamento laterale pesa poco (servono ~40 cm di offset per una riduzione modesta);
la **distanza** è il parametro dominante.

---

## Reiezione dei dark count

Alle HV alte il guadagno esponenziale porta gli impulsi da **singolo fotoelettrone** sopra soglia:
compare una popolazione a bassa carica, 1–2 decadi sotto il picco dei muoni.

**La sottrazione off-time NON è utilizzabile.** Misurato a 1700 V:

| eventi | frazione con impulso sopra soglia nella finestra 250–290 ns |
|---|---|
| **con** muone ($\min \le -100$ mV) | **91.3 %** |
| **senza** muone | 3.3 % |

I conteggi "off-time" sono dominati da **afterpulse e code dell'impulso del muone**, non da dark
current: sottrarli significa sottrarre il segnale.

**Selezione usata**: taglio in carica ancorato alla mediana del run,

$$Q > \frac{Q_{\text{med}}(\mathrm{run})}{\texttt{SIGNAL\_Q\_CUT\_FACTOR}}$$

che segue automaticamente il picco mentre trasla di due decadi con l'HV, invece di restare fisso.

---

## Stima del guadagno

Il valore più probabile della carica è stimato con `MPV_METHOD`:

- `"median"` (default) — mediana con errore da **bootstrap**. Con ~110 eventi/run è l'unico
  stimatore **monotono su tutti i punti** per entrambi i PMT (errori 6–12 %).
- `"logpeak"` — picco di $\log_{10} Q$ con fit gaussiano locale. Instabile dove la distribuzione
  è frastagliata (a ~2.5 eventi/bin il picco può diventare bimodale).
- `"landau"` — fit Moyal. Sconsigliato: produce valori non fisici ($\mathrm{MPV} \sim 10^{-14}$,
  errori $\sim 10^{10}$) perché ampiezza e $\sigma$ non sono vincolate.

La mediana non è l'MPV in senso stretto, ma per la curva di guadagno è equivalente: scala
linearmente col guadagno del PMT.

---

## Turn-on e working point

I conteggi puliti dai dark count, in funzione della HV, si fittano bene con una **sigmoide**
$$n(\mathrm{HV}) = \frac{A}{1 + \exp\!\big(-(\mathrm{HV} - V_{50})/k\big)}$$
con $V_{50} \approx 950\text{–}1025$ V e larghezza $k \approx 35\text{–}45$ V. Il plateau $A$ è
**indipendente dalla soglia** ($-4/-5/-6$ mV danno lo stesso $A$): a plateau ogni muone fa un
impulso enorme. $V_{50}$ sale ~25 V per ogni mV di soglia, e il Top accende ~25 V prima del Bottom
(coerente col suo guadagno ~1.5×). **Working point consigliato: ~1050–1100 V**, appena oltre il
ginocchio, dove si è al 95–99 % del plateau e insensibili alla soglia.

---

## Risultati e bilancio della rate

**Risultati.** I conteggi di segnale (selezione in carica, dark count rimossi) sono **piatti a
plateau (~100 Bottom, ~106 Top su 1000 trigger) da 1000 a 1700 V** per entrambi i PMT: pienamente
efficienti già dalla HV più bassa dello scan. La carica mediana cresce esponenzialmente su due
decadi, Top ~1.5× sopra Bottom.

**Bilancio della rate di coincidenza.** La frazione a plateau è
$$f = \frac{n_\text{segnale}}{N_\text{trigger}} = G \cdot \varepsilon_\text{scan} \cdot \text{(purezza trigger)} \approx 0.103$$
Con $d = 125$ cm il Monte Carlo dà $G = 0.149$ (range $0.117\text{–}0.178$ secondo l'esponente
angolare $\cos^n\theta$, $n = 1\text{–}3$). Quindi **$f/G \approx 0.63\text{–}0.95$**: deficit
del ~5–35 %.

**Cause escluse o limitate:**

| ipotesi | esito |
|---------|-------|
| dark count nel trigger | esclusa: soglia trigger −15 mV ≫ 1 p.e. (~10 mV) |
| disallineamento | escluso: verificato buono |
| distanza sbagliata | esclusa: $d = 125$ cm misurata, già nel conto |
| perdita nella selezione | esclusa: conteggio libero (min $\le -50$ mV su tutta la traccia) $=$ conteggio selezionato |
| attenuazione della luce | **limitata a ≲10 %**: la curva dei conteggi è piatta su un fattore ~60–90 di guadagno (1100→1700 V); una popolazione di muoni a pochi p.e. entrerebbe progressivamente sopra soglia col guadagno, cosa non osservata. Componente in-time a bassa carica: ~20 eventi |
| efficienza intrinseca dei pannelli | **esclusa dalla stima corretta** (sotto): $\varepsilon(\text{pl}) = 1.0\,^{+0}_{-0.25}$ |

**La causa del deficit: trigger sotto plateau.** Ripetendo la curva dei conteggi con soglia
offline **−15 mV** (la stessa del trigger hardware) si misura la curva di efficienza del trigger
stesso: a **1100 V è al ~30 % del plateau** ($r_B = 0.303$, $r_T = 0.335$), e il plateau a quella
soglia arriva solo a **~1250–1300 V**. Il trigger ha quindi lavorato sul fianco ripido del
turn-on. Conseguenze:

1. la componente muonica dei trigger è ~30–50 Hz, sotto le rate misurate (56–88 Hz) → una parte
   dei 1000 trigger **non erano muoni** → $f$ diluita;
2. il campione triggerato è **distorto verso i muoni vicini al PMT di trigger** → $G$ efficace
   ridotta (stessa direzione);
3. l'asimmetria dei singles (87.7 vs 55.7 Hz, rapporto 1.57) è spiegata dal rapporto di guadagno
   ×1.5 dei due PMT, amplificato dal fianco ripido; e il cross-check delle coincidenze torna
   (rapporto misurato 1.49 vs atteso 1.57) → **le durate di logbook sono affidabili** (correzione
   a una conclusione precedente di questa stessa nota).

**Stima dell'efficienza vera dei pannelli** (correzione per il trigger). Le due configurazioni si
mettono a sistema: con $p$ = purezza del trigger,
$$f_B = G\,\varepsilon_B\,p_T,\qquad p_T = \frac{\Phi A\, r_T\,\varepsilon_T}{R_T}
\qquad(\text{e simmetrica scambiando }B \leftrightarrow T)$$
entrambe le equazioni stimano lo stesso prodotto:
$$\varepsilon_B\,\varepsilon_T \;=\; \frac{f_B\,R_T}{G\,\Phi A\,r_T}
\;=\; \frac{f_T\,R_B}{G\,\Phi A\,r_B}$$
Con i valori misurati ($f = 0.109/0.114$, $r = 0.303/0.335$, $R = 87.7/55.7$ Hz) e la griglia di
sistematiche ($G = 0.117\text{–}0.178$, $\Phi A = 70\text{–}100$ Hz, $A = 0.5\ \mathrm{m^2}$):
$\bar\varepsilon = 1.2\text{–}1.7$, con le due equazioni consistenti entro ~35 %. Quindi
$$\varepsilon_\text{pannello}(\text{plateau}) \;=\; 1.0\ ^{+0}_{-0.25}$$
**compatibile con piena efficienza, incompatibile col valore nudo** $f/G \approx 0.74$.
L'overshoot ($>1$) riflette i limiti del modello — flusso ±20 %, esponente angolare, $r$ misurato
sulle coincidenze ma applicato ai singles — non i pannelli.

**Conclusione.** PMT e pannelli funzionano correttamente: turn-on con $V_{50} \approx 1000$ V,
plateau da ~1050 V (soglie −3/−6 mV), efficienza a plateau compatibile con 1 una volta corretto
l'effetto del trigger. L'unico difetto della presa dati è il **working point del trigger**:
1100 V con soglia −15 mV = ~30 % di efficienza, sul fianco del turn-on.

**Raccomandazioni per il re-run:**
1. **trigger a ≥ 1300 V** (il suo plateau a −15 mV): la correzione sparisce e $f$ misura
   direttamente $G\cdot\varepsilon$;
2. **digitizzare anche il PMT di trigger** su un canale libero del digitizer → purezza misurabile
   evento per evento;
3. (opzionale) lettura al centro dei pannelli o ri-wrapping per uniformare la risposta in
   posizione — utile se i pannelli serviranno da telescopio, non necessario per la
   caratterizzazione dei PMT.


In [ ]:
## === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import glob
import gzip
import shutil
import tempfile
import h5py
import pandas as pd
from scipy.signal import iirnotch, filtfilt, find_peaks, butter, savgol_filter
from scipy.optimize import curve_fit
from scipy.stats import moyal


# ============================================================
# RUMORE PERIODICO — FUNZIONI
# ============================================================

def detect_noise_frequencies(wf, fs, n_peaks=6, fmin=5e6, fmax=500e6):
    W = np.fft.rfft(wf)
    freq = np.fft.rfftfreq(len(wf), d=1/fs)
    spec = np.abs(W)

    mask = (freq > fmin) & (freq < fmax)
    freq = freq[mask]
    spec = spec[mask]

    peaks, props = find_peaks(spec, height=np.max(spec) * 0.2, distance=30)

    if len(peaks) == 0:
        return []

    peak_freqs = freq[peaks]
    order = np.argsort(props["peak_heights"])[::-1]
    peak_freqs = peak_freqs[order]

    return peak_freqs[:n_peaks]


def bandstop(wf, fs, f1, f2, order=3):
    b, a = butter(order, [f1/(fs/2), f2/(fs/2)], btype='bandstop')
    return filtfilt(b, a, wf)


def preprocess_waveform(wf_mv, fs, auto_detect=True, max_notch=6, q=60,
                        rf_bands=None, sg_window=9, sg_order=2):
    if rf_bands is None:
        rf_bands = []

    wf_filt = wf_mv.copy()

    if auto_detect:
        freqs = detect_noise_frequencies(wf_mv, fs, n_peaks=max_notch)
    else:
        freqs = []

    for f0 in freqs:
        if f0 < fs / 2:
            b, a = iirnotch(f0, q, fs)
            wf_filt = filtfilt(b, a, wf_filt)

    for f1, f2 in rf_bands:
        if f2 < fs / 2:
            wf_filt = bandstop(wf_filt, fs, f1, f2)

    if sg_window >= 3 and sg_window % 2 == 1 and sg_window < len(wf_filt):
        wf_filt = savgol_filter(wf_filt, window_length=sg_window, polyorder=sg_order)

    return wf_filt


def compute_baseline_and_integral(time_ns, wf_mv, tmin_ns, tmax_ns,
                                  baseline_window_ns=None,
                                  negative_threshold_mv=-5.0):
    signal_mask = (time_ns >= tmin_ns) & (time_ns <= tmax_ns)
    if not np.any(signal_mask):
        raise ValueError(f"Finestra di integrazione vuota: [{tmin_ns}, {tmax_ns}] ns")

    if baseline_window_ns is None:
        baseline_mask = time_ns < tmin_ns
    else:
        b0, b1 = baseline_window_ns
        baseline_mask = (time_ns >= b0) & (time_ns <= b1)

    baseline = np.mean(wf_mv[baseline_mask]) if np.any(baseline_mask) else 0.0
    wf_corr = wf_mv - baseline

    wf_in_window = wf_corr[signal_mask]
    min_in_window = np.min(wf_in_window)
    passed_threshold = min_in_window <= negative_threshold_mv

    if passed_threshold:
        integral_mv_ns = -np.trapezoid(wf_in_window, time_ns[signal_mask])
    else:
        integral_mv_ns = 0.0

    return baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window


# ============================================================
# FIT TIPO LANDAU (approssimazione con Moyal)
# ============================================================

def landau_like_pdf(x, amplitude, mpv, sigma):
    sigma = np.abs(sigma)
    return amplitude * moyal.pdf(x, loc=mpv, scale=sigma)


def fit_landau_histogram(values, nbins=100, fit_range=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if fit_range is not None:
        xmin, xmax = fit_range
        values = values[(values >= xmin) & (values <= xmax)]

    if len(values) < 20:
        return None

    counts, edges = np.histogram(values, bins=nbins, range=fit_range)
    centers = 0.5 * (edges[:-1] + edges[1:])

    mask = counts > 0
    xfit = centers[mask]
    yfit = counts[mask]

    if len(xfit) < 5:
        return None

    amp0 = np.max(yfit)
    mpv0 = xfit[np.argmax(yfit)]
    sigma0 = max(np.std(values) / 5.0, 20.0)

    try:
        popt, pcov = curve_fit(
            landau_like_pdf,
            xfit,
            yfit,
            p0=[amp0, mpv0, sigma0],
            bounds=([0.0, 0.0, 1e-6], [np.inf, np.inf, np.inf]),
            maxfev=20000
        )
        perr = np.sqrt(np.diag(pcov))
    except Exception:
        return None

    return {
        "counts": counts,
        "edges": edges,
        "centers": centers,
        "params": popt,
        "errors": perr,
        "mpv": popt[1],
        "mpv_err": perr[1],
    }


def estimate_mpv_log(values, bins=None, smooth=3, halfwidth=3, max_err_bins=2.0):
    """MPV robusto: picco della distribuzione di log10(Q), raffinato con un fit
    gaussiano locale attorno al massimo.

    Accorgimenti contro l'instabilita' vista con ~100 eventi/run:
      - binning ADATTIVO alla statistica (con 40 bin fissi si hanno ~2 eventi/bin
        e il picco diventa frastagliato/bimodale);
      - il fit raffinato viene ACCETTATO solo se il suo errore e' piccolo rispetto
        alla larghezza di bin, altrimenti si ricade sul massimo dell'istogramma
        lisciato (errore = larghezza di bin / sqrt(12)).
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v) & (v > 0)]
    if len(v) < 20:
        return None

    if bins is None:
        bins = int(np.clip(len(v) / 6.0, 12, 40))   # ~6 eventi per bin

    lg = np.log10(v)
    counts, edges = np.histogram(lg, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bw = float(edges[1] - edges[0])

    if smooth > 1:
        cs = np.convolve(counts.astype(float), np.ones(smooth) / smooth, mode="same")
    else:
        cs = counts.astype(float)

    ipk = int(np.argmax(cs))
    mu, mu_err = float(centers[ipk]), bw / np.sqrt(12.0)

    lo, hi = max(0, ipk - halfwidth), min(len(centers), ipk + halfwidth + 1)
    x, y = centers[lo:hi], counts[lo:hi].astype(float)

    if len(x) >= 4 and y.sum() > 0:
        try:
            popt, pcov = curve_fit(
                lambda xx, A, m, s: A * np.exp(-0.5 * ((xx - m) / s) ** 2),
                x, y,
                p0=[max(y.max(), 1.0), centers[ipk], max((x[-1] - x[0]) / 4, 1e-3)],
                maxfev=10000,
            )
            _e = np.sqrt(np.diag(pcov))
            # accetta solo un picco ben determinato e dentro la finestra fittata
            if (x[0] <= popt[1] <= x[-1] and np.isfinite(_e[1])
                    and _e[1] <= max_err_bins * bw):
                mu = float(popt[1])
                mu_err = max(float(_e[1]), bw / np.sqrt(12.0))
        except Exception:
            pass

    mpv = 10.0 ** mu
    return {"mpv": mpv, "mpv_err": mpv * np.log(10.0) * mu_err,
            "counts": counts, "edges": edges, "n": len(v)}


def estimate_median_charge(values, n_boot=1000, seed=0):
    """Carica MEDIANA con errore da bootstrap.

    Con ~110 eventi/run e' molto piu' stabile del picco: misurato sui dati reali
    risulta monotono su tutti i punti in HV per entrambi i PMT (errori 6-12%),
    mentre il picco di log10(Q) sbanda dove la distribuzione e' frastagliata.
    Non e' l'MPV in senso stretto, ma per la curva di guadagno e' equivalente:
    scala linearmente col guadagno del PMT.
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v) & (v > 0)]
    if len(v) < 20:
        return None
    med = float(np.median(v))
    rng = np.random.default_rng(seed)
    bs = np.median(rng.choice(v, size=(n_boot, len(v)), replace=True), axis=1)
    return {"mpv": med, "mpv_err": float(np.std(bs)), "n": len(v)}


# ============================================================
# UTILITÀ RUN/HV
# ============================================================

def extract_hv_from_filename(filename):
    # HV del PMT SOTTO STUDIO = il token SENZA "TriggExt".
    # Nuovo formato:  ...BottomTriggExt<HVtrig>V_Top<HV>V...  -> studiato = Top
    # Vecchio formato: ...TopTriggExt_Bottom<HV>V...          -> studiato = Bottom
    for key in ("Top", "Bottom"):
        m = re.search(rf'(?<![A-Za-z]){key}(\d{{2,5}})(?:\.\d+)?\s*[Vv]', filename)
        if m:
            return int(m.group(1))
    # Fallback: primo gruppo di 3-5 cifre seguito da V
    m = re.search(r'(\d{3,5})\s*[Vv]', filename)
    if m:
        return int(m.group(1))
    return np.nan


def extract_trigger_hv_from_filename(filename):
    # HV del PMT di TRIGGER (token con "TriggExt"). nan se assente (vecchio formato).
    m = re.search(r'(?:Top|Bottom)TriggExt[_\- ]?(\d{2,5})(?:\.\d+)?\s*[Vv]', filename)
    if m:
        return int(m.group(1))
    return np.nan


def extract_studied_pmt_name(filename):
    # Nome del PMT SOTTO STUDIO (token senza "TriggExt"): "Top" o "Bottom".
    for key in ("Top", "Bottom"):
        if re.search(rf'(?<![A-Za-z]){key}(\d{{2,5}})(?:\.\d+)?\s*[Vv]', filename):
            return key
    return "PMT"


def open_h5_or_gz(file_path):
    if file_path.endswith(".h5"):
        return file_path, None

    if file_path.endswith(".h5.gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False)
        tmp.close()
        with gzip.open(file_path, "rb") as fin, open(tmp.name, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        return tmp.name, tmp.name

    raise ValueError(f"Formato file non supportato: {file_path}")


# ============================================================
# CONFIG
# ============================================================

data_dir = os.path.expanduser("~/Downloads/test_071626")

# Entrambe le configurazioni: Top studiato (trigger Bottom) e Bottom studiato (trigger Top)
RUN_PATTERNS = [
    "MV_PMT_study_BottomTriggExt*Top*V_offset40_0000_2.5Gs_40PT.h5*",   # -> studiato = Top
    "MV_PMT_study_TopTriggExt*Bottom*V_offset40_0000_2.5Gs_40PT.h5*",   # -> studiato = Bottom
]

run_files = sorted({
    p for pat in RUN_PATTERNS for p in glob.glob(os.path.join(data_dir, pat))
})


# Evento e canali da plottare
EVENT_TO_PLOT = 1
# PMT Top = canale 0 (ch1 vuoto, ch8 = rumore costante)
CHANNELS_TO_PLOT = [0]

# --- MODO COMUNE ---
# Il digitizer introduce un modo comune visibile su tutti i canali:
#     signal_corr = ch_studiato - alpha * CM,   CM = mediana dei canali di riferimento
# Da qui in poi TUTTA l'analisi (rate, carica/guadagno) usa signal_corr.
# Misurato su Top@800V (RMS ch0 = 0.736 mV nella regione senza segnale):
#   ch0 - ch1            -> 0.743 mV  (x0.99: ch1 e' piu' rumoroso, non conviene)
#   ch0 - 0.49*ch1       -> 0.609 mV  (x1.21)
#   ch0 - 0.85*med(1..7) -> 0.571 mV  (x1.29)  <- scelta di default
CM_SUBTRACT       = True
CM_REF_CHANNELS   = [1, 2, 3, 4, 5, 6, 7]   # canali vuoti per stimare il modo comune
CM_SCALE          = None                    # None = alpha stimato per evento; 1.0 = sottrazione 1:1
CM_QUIET_WINDOW_NS = (250.0, 400.0)         # regione senza segnale per stimare alpha
# Per la sottrazione secca ch0 - ch1: CM_REF_CHANNELS = [1] e CM_SCALE = 1.0

# --- OPZIONI DEBUG (plot diagnostici, disattivati di default) ---
DEBUG_PLOT_SIGNALS = False   # N segnali/run sovrapposti, raggruppati per HV
DEBUG_PLOT_AVG     = False   # segnale medio per run (rumore vs impulso coerente)
DEBUG_PLOT_CHANNELS = False   # primi N canali, griglia 2x4, una figura per evento
PLOT_CHARGE            = True   # carica: fit Landau-like + MPV vs HV (studio del guadagno)
DEBUG_PLOT_CHARGE_HIST = True   # griglia delle distribuzioni di carica (righe=HV, colonne=PMT)
DEBUG_PLOT_CHARGE_HIST_SINGLE = False  # in piu': una figura separata per ogni run (38 figure)

# Istogrammi di carica: range/binning INDIPENDENTI da FIT_RANGE, per vedere
# la distribuzione completa e scegliere poi il range del fit.
CHARGE_HIST_RANGE = None   # None = auto (percentili 0.5-99.5 su tutti i run)
CHARGE_HIST_LOGX  = True   # scala x logaritmica (comune a TUTTI i pannelli)
CHARGE_HIST_LOGY  = False  # scala y lineare
# Il piedistallo ha ~1000 eventi contro i ~110 del segnale: in scala lineare lo
# schiaccerebbe. True = lo riscala all'altezza del segnale (conta la POSIZIONE,
# non l'altezza); False = conteggi grezzi.
CHARGE_HIST_PED_SCALE = True
# Decadi di margine a destra: il limite superiore va preso sulla coda dei run a
# HV ALTA, altrimenti un percentile globale (dominato dalle HV intermedie)
# taglia le distribuzioni a 1650-1700 V.
CHARGE_HIST_XMARGIN_DEC = 1.0
CHARGE_HIST_BINS  = 60

# Parametri del plot per-evento dei canali
CH_PLOT_CHANNELS   = list(range(8))  # primi 8 canali -> griglia 2 righe x 4 colonne
CH_PLOT_YLIM       = (-5.0, 5.0)     # range asse Y [mV]
CH_PLOT_N_EVENTS   = 3               # eventi da plottare per ciascun run
CH_PLOT_RUN_FILTER = None            # es. "Top1700" per un solo run; None = tutti
CH_PLOT_MAX_FIGS   = 12              # tetto totale di figure (evita di saturare il notebook)
# Limiti asse Y [mV] per il plot dei segnali: gli impulsi arrivano a ~-800 mV
# alle HV alte e schiaccerebbero tutto il resto. None = autoscala su quel lato.
SIGNAL_YLIM_BOTTOM_MV = -20
SIGNAL_YLIM_TOP_MV    = 5

# Numero di segnali per run da mostrare (evoluzione vs HV)
N_SIGNALS_TO_PLOT = 1

# Limite eventi analizzati; None = tutti
MAX_EVENTS_FOR_ANALYSIS = None

# Finestra integrazione [ns]
# Finestra di integrazione = finestra fisica del segnale (i muoni stanno tutti
# in 80-120 ns). Stringerla evita di integrare 400 ns di rumore e libera spazio
# per una finestra di piedistallo della STESSA larghezza.
INTEGRATION_WINDOW_NS = (80.0, 120.0)

# Piedistallo: stessa larghezza (40 ns), regione senza segnale -> dice dove sta
# lo zero e quanto vale il rumore di integrazione. Calcolato per TUTTI gli eventi.
PEDESTAL_WINDOW_NS = (250.0, 290.0)

# Finestra baseline [ns]; se None usa tutti i campioni prima di tmin
BASELINE_WINDOW_NS = (0.0, 40.0)

# Soglia negativa [mV]: l'integrale viene fatto solo se il minimo nella finestra è <= soglia
NEGATIVE_THRESHOLD_MV = -3.0

# Normalizzazione della curva di rate: False = conteggi grezzi per run
# (i run hanno tutti 1000 trigger, quindi i conteggi sono gia' confrontabili).
# True = divide per la durata del run e mostra Hz.
RATE_NORMALIZE_BY_DURATION = False

# Conteggi di segnale selezionati dallo SPETTRO DI CARICA.
# NB: la sottrazione off-time non e' utilizzabile: a HV alta il 91% degli eventi
# con muone ha un impulso sopra soglia anche nella finestra di piedistallo
# (afterpulse/coda del segnale), contro il 3% degli eventi senza muone.
# Si taglia invece in carica: q_cut = mediana_del_run / SIGNAL_Q_CUT_FACTOR,
# che segue automaticamente il picco mentre trasla di decadi con l'HV.
PLOT_SIGNAL_COUNTS  = True
SIGNAL_Q_CUT_FACTOR = 5.0

# Conteggi vs HV a piu' soglie di ampiezza (min nel gate <= soglia), 2 PMT sovrapposti
PLOT_COUNTS_VS_THRESHOLD = True
SIGNAL_THRESHOLDS_MV     = [-4.0, -5.0, -6.0]
FIT_COUNTS_SIGMOID       = True   # fit sigmoide dei conteggi vs HV

# --- REIEZIONE DARK COUNT ---
# I muoni sono agganciati al trigger (t_min = 96-100 ns, IQR ~12 ns); i dark count
# sono uniformi sulla traccia e hanno ampiezza appena sopra soglia (~-10 mV).
# Il taglio temporale e' il discriminante pulito fra le due popolazioni.
SIGNAL_TIME_WINDOW_NS = (80.0, 120.0)

# Range del fit Landau: il picco trasla di ~3 decadi con l'HV, quindi un range
# fisso non puo' funzionare -> per default si adatta run per run sui percentili.
MPV_METHOD = "median"         # "median" = mediana + bootstrap (piu' stabile, default)
                              # "logpeak" = picco di log10(Q) | "landau" = fit Moyal
FIT_RANGE_MODE = "auto"       # "auto" = percentili per-run; "fixed" = usa FIT_RANGE
FIT_RANGE_PCT  = (2.0, 99.0)  # percentili usati in modalita' auto

# Finestra di CONTEGGIO per la rate = gate dell'ADC. Il conteggio degli eventi
# sopra soglia usa SOLO questa finestra. Default = finestra di integrazione.
COUNT_WINDOW_NS = INTEGRATION_WINDOW_NS

# Durata del run [s] per convertire i conteggi in rate [Hz].
# NB: NON è presente nell'header dei .h5 (verificato: nessun timestamp).
# Impostala a mano se nota (o metti il trigger rate), altrimenti resta in conteggi.
# Durata SPECIFICA di ogni run [s], per PMT SOTTO STUDIO e HV.
# Trascritte dal logbook: pagina "MISURE PMT / Bottom" e pagina "MISURE PMTS / TOP".
# NB: il logbook va a passi di 50 V (800-1700 V), i file a passi di 100 V e
#     includono 500/600/700 V -> per quelle HV la durata NON e' disponibile.
RUN_DURATION_S_BY_PMT = {
    "Bottom": {
        800: 11.399020,  850: 11.454379,  900: 11.476146,  950: 11.483679,
        1000: 11.379606, 1050: 11.407868, 1100: 11.228207, 1150: 11.433082,
        1200: 11.398867, 1250: 11.316560, 1300: 11.423236, 1350: 11.462118,
        1400: 11.372692, 1450: 11.365217, 1500: 11.321387, 1550: 11.349531,
        1600: 11.459816, 1650: 11.395518, 1700: 11.486925,
    },
    "Top": {
        800: 18.319671,  850: 19.346358,  900: 19.310514,  950: 17.291294,
        1000: 18.339861, 1050: 18.353394, 1100: 17.565702, 1150: 19.549154,
        1200: 18.412251, 1250: 18.358188, 1300: 17.480076, 1350: 17.435875,
        1400: 17.246767, 1450: 17.369507, 1500: 18.322093, 1550: 16.453156,
        1600: 18.329035, 1650: 17.557127, 1700: 16.367028,
    },
}

# Canali da leggere per ciascuna configurazione (PMT sotto studio -> canali).
# Si usa SEMPRE lo stesso ingresso del digitizer (ch0) per il PMT sotto studio:
# in un caso e' il PMT Top, nell'altro il PMT Bottom.
# Verificato su test_071626: entrambi danno segnale su ch0 e scalano con l'HV
# (Top: -21 mV @800V -> -804 mV @1700V; Bottom: -4.9 mV @800V -> -804 mV @1700V).
CHANNELS_BY_STUDIED = {"Top": [0], "Bottom": [0]}

# Durata usata se la HV non e' nel dizionario: NaN => il run viene escluso
# dalla curva in Hz (e segnalato), invece di essere normalizzato a caso.
RUN_DURATION_FALLBACK_S = np.nan

Q = 60
MAX_NOTCH = 6
AUTO_DETECT = True

RF_BANDS = [
    (80e6, 95e6),
    (120e6, 135e6),
    (180e6, 200e6),
    (240e6, 260e6),
]

SG_WINDOW = 9
SG_ORDER = 2

ADC_TO_MV = 1000.0 / 4096.0
NOMINAL_CHANNELS = 32

# Parametri istogramma/fit
HIST_BINS = 100
FIT_RANGE = (400, 5000)
HIST_XLIM = (400, 5000)

# File output
RESULTS_CSV = "charge_results_by_event.csv"
MPV_CSV = "mpv_landau_fit_results.csv"


# ============================================================
# CHECK FILE
# ============================================================

if len(run_files) == 0:
    raise FileNotFoundError(
        f"Nessun file trovato in {data_dir} con pattern:\n"
        "MV_PMT_study_TopTriggExt_Bottom*V_offset40_0000_2.5Gs_40PT.h5.gz"
    )

missing = [p for p in run_files if not os.path.exists(p)]

if missing:
    raise FileNotFoundError("File mancanti:\n" + "\n".join(missing))

print("Run trovati:")
for f in run_files:
    _bn = os.path.basename(f)
    _hv = extract_hv_from_filename(_bn)
    _hvt = extract_trigger_hv_from_filename(_bn)
    _flag = "  <-- HV NON PARSATA" if not np.isfinite(_hv) else ""
    print(f"   HV_PMT={_hv} V | HV_trig={_hvt} V  {_bn}{_flag}")


# ============================================================
# ANALISI MULTI-RUN
# ============================================================

all_results = []
plot_cache = []
signal_plot_cache = []
ch_plot_cache = []
avg_accum = {}  # (run, channel) -> dict(sum, count, hv_V, time_ns)

for original_file_path in run_files:
    base_name = os.path.basename(original_file_path)
    run_label = base_name.replace(".h5.gz", "").replace(".h5", "")
    hv_value = extract_hv_from_filename(base_name)
    trigger_hv_value = extract_trigger_hv_from_filename(base_name)
    studied_pmt = extract_studied_pmt_name(base_name)
    run_duration_s = RUN_DURATION_S_BY_PMT.get(studied_pmt, {}).get(hv_value, RUN_DURATION_FALLBACK_S)
    channels_this_run = CHANNELS_BY_STUDIED.get(studied_pmt, CHANNELS_TO_PLOT)

    print(f"\n=== Analizzo {run_label} | studiato = {studied_pmt} | HV = {hv_value} V | HV_trig = {trigger_hv_value} V | dt = {run_duration_s} s ===")

    file_path, temp_file_to_remove = open_h5_or_gz(original_file_path)

    try:
        with h5py.File(file_path, "r") as f:
            config = f["/config"]
            sampling_time = float(config.attrs["SamplingTime"])
            fs = 1.0 / sampling_time

            events_group = f["/events"]
            event_keys = sorted(events_group.keys(), key=lambda x: int(x.replace("event", "")))

            if MAX_EVENTS_FOR_ANALYSIS is not None:
                event_keys = event_keys[:MAX_EVENTS_FOR_ANALYSIS]

            first = np.array(events_group[event_keys[0]][:], dtype=np.float64)
            total_samples = len(first)
            samples_per_channel = total_samples // NOMINAL_CHANNELS
            time_axis = np.arange(samples_per_channel) * sampling_time * 1e9

            for ev_index, ev_name in enumerate(event_keys):
                data_corr = np.array(events_group[ev_name][:], dtype=np.float64)

                if len(data_corr) != samples_per_channel * NOMINAL_CHANNELS:
                    print(f"⚠️ Evento corrotto, skip: {ev_name}")
                    continue

                data_mat = data_corr.reshape((NOMINAL_CHANNELS, samples_per_channel))

                # Debug: salva i primi canali di alcuni eventi (baseline sottratta, non filtrati)
                if (DEBUG_PLOT_CHANNELS
                        and ev_index < CH_PLOT_N_EVENTS
                        and len(ch_plot_cache) < CH_PLOT_MAX_FIGS
                        and (CH_PLOT_RUN_FILTER is None or CH_PLOT_RUN_FILTER in run_label)):
                    _bm = (time_axis >= BASELINE_WINDOW_NS[0]) & (time_axis <= BASELINE_WINDOW_NS[1])
                    _wfs = {}
                    for _c in CH_PLOT_CHANNELS:
                        _w = data_mat[_c] * ADC_TO_MV
                        _wfs[_c] = _w - (np.mean(_w[_bm]) if np.any(_bm) else 0.0)
                    ch_plot_cache.append({
                        "run": run_label, "studied": studied_pmt, "hv_V": hv_value,
                        "event": ev_index, "time_ns": time_axis.copy(), "wfs": _wfs,
                    })

                for ch in channels_this_run:
                    _raw = data_mat[ch] * ADC_TO_MV

                    # signal_corr = ch - alpha * CM  (rimuove il modo comune del digitizer)
                    _refs = [c for c in CM_REF_CHANNELS if c != ch] if CM_SUBTRACT else []
                    if _refs:
                        _cm = np.median(data_mat[_refs, :], axis=0) * ADC_TO_MV
                        if CM_SCALE is None:
                            _qm = ((time_axis >= CM_QUIET_WINDOW_NS[0]) &
                                   (time_axis <= CM_QUIET_WINDOW_NS[1]))
                            _c = _cm[_qm] - _cm[_qm].mean()
                            _s = _raw[_qm] - _raw[_qm].mean()
                            _den = float(np.dot(_c, _c))
                            _alpha = float(np.dot(_s, _c) / _den) if _den > 0 else 0.0
                        else:
                            _alpha = float(CM_SCALE)
                        wf_mv = _raw - _alpha * _cm
                    else:
                        wf_mv = _raw

                    wf_filt = preprocess_waveform(
                        wf_mv, fs,
                        auto_detect=AUTO_DETECT,
                        max_notch=MAX_NOTCH,
                        q=Q,
                        rf_bands=RF_BANDS,
                        sg_window=SG_WINDOW,
                        sg_order=SG_ORDER,
                    )

                    baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window = compute_baseline_and_integral(
                        time_axis,
                        wf_filt,
                        INTEGRATION_WINDOW_NS[0],
                        INTEGRATION_WINDOW_NS[1],
                        baseline_window_ns=BASELINE_WINDOW_NS,
                        negative_threshold_mv=NEGATIVE_THRESHOLD_MV,
                    )

                    # Conteggio rate: soglia valutata SOLO nel gate ADC (COUNT_WINDOW_NS)
                    # Piedistallo: integrale su finestra senza segnale, SENZA condizioni
                    _pm = ((time_axis >= PEDESTAL_WINDOW_NS[0])
                           & (time_axis <= PEDESTAL_WINDOW_NS[1]))
                    ped_mV_ns = (float(-np.trapezoid(wf_corr[_pm], time_axis[_pm]))
                                 if np.any(_pm) else np.nan)
                    # stessa soglia nella finestra off-time -> misura dei dark count
                    passed_ped = (bool(np.min(wf_corr[_pm]) <= NEGATIVE_THRESHOLD_MV)
                                  if np.any(_pm) else False)

                    count_mask = (time_axis >= COUNT_WINDOW_NS[0]) & (time_axis <= COUNT_WINDOW_NS[1])
                    if np.any(count_mask):
                        _cw = wf_corr[count_mask]
                        _ct = time_axis[count_mask]
                        _i = int(np.argmin(_cw))
                        min_in_count_mV = float(_cw[_i])
                        t_min_ns = float(_ct[_i])          # tempo dell'impulso
                        passed_count = bool(min_in_count_mV <= NEGATIVE_THRESHOLD_MV)
                    else:
                        min_in_count_mV, t_min_ns, passed_count = np.nan, np.nan, False

                    all_results.append({
                        "run": run_label,
                        "studied": studied_pmt,
                        "hv_V": hv_value,
                        "hv_trig_V": trigger_hv_value,
                        "event": ev_index,
                        "channel": ch,
                        "baseline_mV": baseline,
                        "min_in_window_mV": min_in_window,
                        "passed_threshold": passed_threshold,
                        "passed_count": passed_count,
                        "t_min_ns": t_min_ns,
                        "ped_mV_ns": ped_mV_ns,
                        "passed_ped": passed_ped,
                        "min_in_count_mV": min_in_count_mV,
                        "run_duration_s": run_duration_s,
                        "integral_mV_ns": integral_mv_ns,
                    })

                    key = (run_label, ch)
                    acc = avg_accum.get(key)
                    if acc is None:
                        avg_accum[key] = {
                            "sum": wf_corr.copy(),
                            "count": 1,
                            "studied": studied_pmt,
                            "hv_V": hv_value,
                            "hv_trig_V": trigger_hv_value,
                            "time_ns": time_axis.copy(),
                        }
                    else:
                        acc["sum"] += wf_corr
                        acc["count"] += 1

                    if DEBUG_PLOT_SIGNALS and ev_index < N_SIGNALS_TO_PLOT:
                        signal_plot_cache.append({
                            "run": run_label,
                            "studied": studied_pmt,
                            "hv_V": hv_value,
                            "hv_trig_V": trigger_hv_value,
                            "event": ev_index,
                            "channel": ch,
                            "time_ns": time_axis.copy(),
                            "wf_corr": wf_corr.copy(),
                        })

                    if ev_index == EVENT_TO_PLOT and ch in CHANNELS_TO_PLOT:
                        plot_cache.append({
                            "run": run_label,
                            "hv_V": hv_value,
                            "event": ev_index,
                            "channel": ch,
                            "time_ns": time_axis.copy(),
                            "wf_filt": wf_filt.copy(),
                            "wf_corr": wf_corr.copy(),
                            "signal_mask": signal_mask.copy(),
                            "baseline_mask": baseline_mask.copy(),
                            "integral_mV_ns": integral_mv_ns,
                            "passed_threshold": passed_threshold,
                            "min_in_window_mV": min_in_window,
                        })
    finally:
        if temp_file_to_remove is not None and os.path.exists(temp_file_to_remove):
            os.remove(temp_file_to_remove)


# ============================================================
# TABELLE RISULTATI
# ============================================================

df = pd.DataFrame(all_results)

summary = (
    df.groupby(["run", "hv_V", "channel"])
      .agg(
          n_entries=("integral_mV_ns", "count"),
          n_over_threshold=("passed_threshold", "sum"),
          mean_integral_mV_ns=("integral_mV_ns", "mean"),
          std_integral_mV_ns=("integral_mV_ns", "std"),
          min_integral_mV_ns=("integral_mV_ns", "min"),
          max_integral_mV_ns=("integral_mV_ns", "max"),
      )
      .sort_index()
)

display(summary)
display(df.head())

df.to_csv(RESULTS_CSV, index=False)
print(f"Salvato file risultati evento-per-evento: {RESULTS_CSV}")


# ============================================================
# Plot dei segnali: attivabili con DEBUG_PLOT_SIGNALS / DEBUG_PLOT_AVG.
# Restano commentati i plot ADC (istogrammi carica + Landau) e MPV vs HV.
# ============================================================

# ============================================================
# DEBUG — primi canali per evento (griglia 2x4)
# ============================================================

if DEBUG_PLOT_CHANNELS:
    if len(ch_plot_cache) == 0:
        print("Nessun evento in cache per il plot dei canali.")

    _ncol = 4
    _nrow = int(np.ceil(len(CH_PLOT_CHANNELS) / _ncol))

    for item in ch_plot_cache:
        fig, axes = plt.subplots(
            _nrow, _ncol, figsize=(4.0 * _ncol, 2.6 * _nrow),
            sharex=True, sharey=True, squeeze=False,
        )

        for k, cch in enumerate(CH_PLOT_CHANNELS):
            ax = axes[k // _ncol][k % _ncol]
            ax.plot(item["time_ns"], item["wfs"][cch], lw=0.8, color="crimson")
            ax.axvspan(COUNT_WINDOW_NS[0], COUNT_WINDOW_NS[1], color="orange", alpha=0.15)
            ax.axhline(0.0, color="black", lw=0.6, ls="--")
            ax.axhline(NEGATIVE_THRESHOLD_MV, color="navy", lw=0.8, ls=":")
            ax.set_ylim(CH_PLOT_YLIM)
            ax.set_title(f"ch {cch}", fontsize=10)
            ax.grid(True, alpha=0.25)

        # nasconde eventuali pannelli in eccesso
        for k in range(len(CH_PLOT_CHANNELS), _nrow * _ncol):
            axes[k // _ncol][k % _ncol].axis("off")

        for j in range(_ncol):
            axes[-1][j].set_xlabel("Time [ns]")
        for i in range(_nrow):
            axes[i][0].set_ylabel("Amplitude [mV]")

        fig.suptitle(
            f"{item['studied']} @ {item['hv_V']} V \u2014 event {item['event']} "
            f"(baseline subtracted, unfiltered)",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()


if DEBUG_PLOT_SIGNALS:
    # ============================================================
    # PLOT DEI SEGNALI — 10 segnali per run, raggruppati per HV
    # ============================================================

    if len(signal_plot_cache) == 0:
        print("Nessun segnale disponibile per il plot.")
    else:
        # Righe = run ordinati per HV crescente; colonne = canale
        runs_by_hv = sorted(
            set((it["run"], it["hv_V"], it["hv_trig_V"], it["studied"])
                for it in signal_plot_cache),
            # raggruppa per PMT studiato, poi HV crescente
            key=lambda rh: (rh[3], np.inf if not np.isfinite(rh[1]) else rh[1])
        )
        channels = sorted(set(it["channel"] for it in signal_plot_cache))

        nrow = len(runs_by_hv)
        ncol = len(channels)

        fig, axes = plt.subplots(
            nrow, ncol,
            figsize=(6.5 * ncol, 2.6 * nrow),
            sharex=True, sharey=True,   # asse Y condiviso: evoluzione ampiezza visibile
            squeeze=False,
        )

        for i, (run_label_i, hv_i, hv_trig_i, studied_i) in enumerate(runs_by_hv):
            for j, ch in enumerate(channels):
                ax = axes[i][j]

                sigs = [it for it in signal_plot_cache
                        if it["run"] == run_label_i and it["channel"] == ch]
                sigs = sorted(sigs, key=lambda it: it["event"])[:N_SIGNALS_TO_PLOT]

                for _k, it in enumerate(sigs):
                    ax.plot(it["time_ns"], it["wf_corr"], lw=0.8, alpha=0.7,
                            label=(f"PMT {studied_i} (ch{ch}) @ {hv_i} V" if _k == 0 else None))

                ax.legend(loc="upper right", fontsize=7)

                ax.axvspan(INTEGRATION_WINDOW_NS[0], INTEGRATION_WINDOW_NS[1],
                           color="orange", alpha=0.15)
                ax.axhline(0.0, color="black", lw=0.6, ls="--")
                ax.axhline(NEGATIVE_THRESHOLD_MV, color="navy", lw=0.8, ls=":")
                ax.grid(True, alpha=0.25)

                if i == 0:
                    ax.set_title(f"Channel {ch}")
                if j == 0:
                    ax.set_ylabel(f"{studied_i} {hv_i} V\n(trig {hv_trig_i} V)\n[mV]", fontsize=9)

        # Scala verticale limitata (sharey=True -> vale per tutti i pannelli)
        if SIGNAL_YLIM_BOTTOM_MV is not None:
            axes[0][0].set_ylim(bottom=SIGNAL_YLIM_BOTTOM_MV)
        if SIGNAL_YLIM_TOP_MV is not None:
            axes[0][0].set_ylim(top=SIGNAL_YLIM_TOP_MV)

        for j in range(ncol):
            axes[-1][j].set_xlabel("Time [ns]")

        fig.suptitle(
            f"PMT signal evolution vs HV ({N_SIGNALS_TO_PLOT} signals/run)",
            y=1.001, fontsize=13
        )
        plt.tight_layout()
        plt.show()


if DEBUG_PLOT_AVG:
    # ============================================================
    # PLOT DELLE MEDIE DEI SEGNALI — una media per run, HV in etichetta
    # ============================================================

    if len(avg_accum) == 0:
        print("Nessuna media disponibile per il plot.")
    else:
        # Righe = run ordinati per HV crescente; colonne = canale
        runs_by_hv = sorted(
            set((run_i, acc["hv_V"], acc["hv_trig_V"], acc["studied"])
                for (run_i, ch_i), acc in avg_accum.items()),
            key=lambda rh: (rh[3], np.inf if not np.isfinite(rh[1]) else rh[1])
        )
        channels = sorted(set(ch_i for (run_i, ch_i) in avg_accum.keys()))

        nrow = len(runs_by_hv)
        ncol = len(channels)

        fig, axes = plt.subplots(
            nrow, ncol,
            figsize=(6.5 * ncol, 2.6 * nrow),
            sharex=True, sharey=True,   # asse Y condiviso: confronto ampiezza medie vs HV
            squeeze=False,
        )

        for i, (run_label_i, hv_i, hv_trig_i, studied_i) in enumerate(runs_by_hv):
            for j, ch in enumerate(channels):
                ax = axes[i][j]
                acc = avg_accum.get((run_label_i, ch))

                if acc is not None and acc["count"] > 0:
                    wf_mean = acc["sum"] / acc["count"]
                    ax.plot(acc["time_ns"], wf_mean, lw=1.3, color="crimson",
                            label=f"PMT {studied_i} (ch{ch}) @ {hv_i} V")
                    ax.legend(loc="upper right", fontsize=7)
                    vmin = np.min(wf_mean[
                        (acc["time_ns"] >= INTEGRATION_WINDOW_NS[0]) &
                        (acc["time_ns"] <= INTEGRATION_WINDOW_NS[1])
                    ])
                    # HV scritto nel plot + minimo medio nella finestra
                    ax.text(
                        0.02, 0.05,
                        f"HV = {hv_i} V (trig {hv_trig_i} V)\nN = {acc['count']}\nmin = {vmin:.2f} mV",
                        transform=ax.transAxes, ha="left", va="bottom", fontsize=9,
                        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85),
                    )

                ax.axvspan(INTEGRATION_WINDOW_NS[0], INTEGRATION_WINDOW_NS[1],
                           color="orange", alpha=0.15)
                ax.axhline(0.0, color="black", lw=0.6, ls="--")
                ax.axhline(NEGATIVE_THRESHOLD_MV, color="navy", lw=0.8, ls=":")
                ax.grid(True, alpha=0.25)

                if i == 0:
                    ax.set_title(f"Channel {ch}")
                if j == 0:
                    ax.set_ylabel(f"{studied_i} {hv_i} V\n(trig {hv_trig_i} V)\n[mV]", fontsize=9)

        for j in range(ncol):
            axes[-1][j].set_xlabel("Time [ns]")

        fig.suptitle("Average PMT signal per run (mean over all events)",
                     y=1.001, fontsize=13)
        plt.tight_layout()
        plt.show()


# (istogrammi di carica, fit Landau e MPV vs HV: vedi sezione CARICA in fondo)


# ============================================================
# CURVE DI RATE SOVRAPPOSTE — Top studiato vs Bottom studiato
# ============================================================
# Conteggio ristretto al gate ADC (COUNT_WINDOW_NS) tramite 'passed_count'.
# Normalizzazione con la durata SPECIFICA di ogni run (RUN_DURATION_S_BY_PMT).
# Asse X = HV del PMT sotto studio -> le due configurazioni sono confrontabili.

TRIG_OF = {"Top": "Bottom", "Bottom": "Top"}
COLORS  = {"Top": "tab:blue", "Bottom": "tab:red"}
MARKERS = {"Top": "o", "Bottom": "s"}

rate_df = (
    df.groupby(["studied", "channel", "hv_V"])
      .agg(
          n_over=("passed_count", "sum"),
          n_tot=("passed_count", "count"),
          hv_trig_V=("hv_trig_V", "first"),
          dur_s=("run_duration_s", "first"),
      )
      .reset_index()
      .sort_values(["studied", "channel", "hv_V"])
)

rate_df["frac"] = rate_df["n_over"] / rate_df["n_tot"]
rate_df["rate_Hz"] = rate_df["n_over"] / rate_df["dur_s"]
rate_df["rate_err_Hz"] = np.sqrt(rate_df["n_over"]) / rate_df["dur_s"]

dur_ok = np.isfinite(rate_df["dur_s"].to_numpy(dtype=float))

if RATE_NORMALIZE_BY_DURATION:
    for st in sorted(rate_df["studied"].unique()):
        miss = sorted(set(rate_df.loc[(~dur_ok) & (rate_df["studied"] == st), "hv_V"]))
        if miss:
            print(f"[ATTENZIONE] {st}: durata non disponibile per HV = {miss} V "
                  f"-> punti esclusi dalla curva in Hz")

if RATE_NORMALIZE_BY_DURATION and dur_ok.any():
    ycol, yerrcol = "rate_Hz", "rate_err_Hz"
    ylabel = "Rate [Hz]"
    _d = rate_df["dur_s"].to_numpy(dtype=float)[dur_ok]
    rate_note = f"per-run durations from logbook ({_d.min():.2f}\u2013{_d.max():.2f} s)"
else:
    rate_df["n_over_err"] = np.sqrt(rate_df["n_over"])
    ycol, yerrcol = "n_over", "n_over_err"
    _ntrig = int(rate_df["n_tot"].iloc[0]) if len(rate_df) else 0
    ylabel = f"Counts above threshold (out of {_ntrig} triggers/run)"
    rate_note = "raw counts, not normalized to run duration"

fig, ax = plt.subplots(figsize=(9.0, 6.0))

for st in sorted(rate_df["studied"].unique()):
    for ch in sorted(rate_df.loc[rate_df["studied"] == st, "channel"].unique()):
        sel = rate_df[(rate_df["studied"] == st) & (rate_df["channel"] == ch)].sort_values("hv_V")
        ax.errorbar(
            sel["hv_V"], sel[ycol], yerr=sel[yerrcol],
            marker=MARKERS.get(st, "o"), linestyle="-", linewidth=1.6, capsize=3,
            color=COLORS.get(st), 
            label=f"{st} under test (ch{ch}) \u2014 trigger {TRIG_OF.get(st, '?')}",
        )

ax.set_xlabel("HV of PMT under test [V]")
ax.set_ylabel(ylabel)
ax.set_title(
    f"Rate curves \u2014 Top vs Bottom | gate {COUNT_WINDOW_NS[0]:.0f}\u2013{COUNT_WINDOW_NS[1]:.0f} ns, "
    f"threshold {NEGATIVE_THRESHOLD_MV:.1f} mV\n({rate_note})",
    fontsize=10,
)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

display(rate_df)
rate_df.to_csv("rate_curve_by_channel.csv", index=False)
print("Salvata curva di rate: rate_curve_by_channel.csv")


# ============================================================
# SIGNAL COUNTS — charge-selected (dark counts rejected), vs HV
# ============================================================
# I dark count formano una popolazione a bassa carica, ~1-2 decadi sotto il picco
# dei muoni (visibile negli istogrammi di carica). La soglia si ancora alla mediana
# del run, cosi' segue il picco che trasla con l'HV invece di restare fissa.

if PLOT_SIGNAL_COUNTS:
    _in = (df["passed_count"]
           & df["t_min_ns"].between(*SIGNAL_TIME_WINDOW_NS)
           & np.isfinite(df["integral_mV_ns"])
           & (df["integral_mV_ns"] > 0))

    _d = df.loc[_in, ["studied", "channel", "hv_V", "integral_mV_ns"]].copy()
    _key = ["studied", "channel", "hv_V"]

    # soglia in carica ancorata alla mediana del run
    _d["q_cut"] = _d.groupby(_key)["integral_mV_ns"].transform("median") / SIGNAL_Q_CUT_FACTOR
    _d["is_signal"] = _d["integral_mV_ns"] > _d["q_cut"]

    sig_df = (
        _d.groupby(_key)
          .agg(n_in=("is_signal", "count"),
               n_signal=("is_signal", "sum"),
               q_med=("integral_mV_ns", "median"),
               q_cut=("q_cut", "first"))
          .reset_index()
          .sort_values(_key)
    )
    sig_df["n_lowq"] = sig_df["n_in"] - sig_df["n_signal"]      # dark count residui
    sig_df["n_signal_err"] = np.sqrt(sig_df["n_signal"])
    _ntrig = int(df.groupby(_key)["passed_count"].count().max())

    fig, ax = plt.subplots(figsize=(9.0, 6.0))

    for st in sorted(sig_df["studied"].unique()):
        for ch in sorted(sig_df.loc[sig_df["studied"] == st, "channel"].unique()):
            sel = sig_df[(sig_df["studied"] == st) & (sig_df["channel"] == ch)].sort_values("hv_V")
            _c = COLORS.get(st, "tab:blue")
            ax.errorbar(
                sel["hv_V"], sel["n_signal"], yerr=sel["n_signal_err"],
                marker=MARKERS.get(st, "o"), linestyle="-", linewidth=1.8, capsize=3,
                color=_c, label=f"{st} under test (ch{ch})",
            )
            ax.plot(sel["hv_V"], sel["n_lowq"], linestyle="--", linewidth=1.0,
                    marker=".", markersize=4, color=_c, alpha=0.5,
                    label=f"{st} \u2014 low-charge (dark counts)")

    ax.set_xlabel("HV of PMT under test [V]")
    ax.set_ylabel(f"Signal events (out of {_ntrig} triggers/run)")
    ax.set_title(
        f"Signal events vs HV \u2014 charge-selected, dark counts rejected\n"
        f"(gate {COUNT_WINDOW_NS[0]:.0f}\u2013{COUNT_WINDOW_NS[1]:.0f} ns, "
        f"Q > median/{SIGNAL_Q_CUT_FACTOR:g})",
        fontsize=10,
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    display(sig_df)
    sig_df.to_csv("signal_counts_charge_selected.csv", index=False)
    print("Salvati conteggi di segnale: signal_counts_charge_selected.csv")


# ============================================================
# COUNTS vs HV — a piu' soglie di ampiezza, conteggi PULITI dai dark count
# ============================================================
# Un evento conta se supera la soglia di ampiezza nel gate (min_in_count_mV <= soglia)
# E ha carica sopra il taglio (integral > q_cut), cioe' NON e' nella popolazione a
# bassa carica dei dark count. q_cut = mediana_del_run / SIGNAL_Q_CUT_FACTOR, la stessa
# usata nei conteggi charge-selected. (La sottrazione off-time non e' utilizzabile:
# a HV alta e' dominata da afterpulse/code del muone, non da dark current.)

if PLOT_COUNTS_VS_THRESHOLD:
    _thr_styles = ["-", "--", ":"]
    _thr_marks = ["o", "s", "^"]
    _key = ["studied", "channel", "hv_V"]

    # taglio in carica per run, ancorato alla mediana degli eventi in tempo
    _base = df[df["passed_count"]
               & df["t_min_ns"].between(*SIGNAL_TIME_WINDOW_NS)
               & np.isfinite(df["integral_mV_ns"]) & (df["integral_mV_ns"] > 0)]
    _qcut = (_base.groupby(_key)["integral_mV_ns"].median() / SIGNAL_Q_CUT_FACTOR)

    _valid = df["min_in_count_mV"].notna()
    rows = []
    for (st, ch, hv), g in df[_valid].groupby(_key):
        m = g["min_in_count_mV"].to_numpy(dtype=float)
        q = g["integral_mV_ns"].to_numpy(dtype=float)
        qc = float(_qcut.get((st, ch, hv), np.nan))
        clean = np.isfinite(q) & (q > qc)   # non dark count
        row = {"studied": st, "channel": ch, "hv_V": hv, "n_tot": len(m), "q_cut": qc}
        for thr in SIGNAL_THRESHOLDS_MV:
            row[f"n_{abs(thr):.0f}mV"] = int(((m <= thr) & clean).sum())
        rows.append(row)
    thr_df = pd.DataFrame(rows).sort_values(_key)
    _ntrig = int(thr_df["n_tot"].iloc[0]) if len(thr_df) else 0

    def _sigmoid(hv, A, v50, k):
        return A / (1.0 + np.exp(-(hv - v50) / k))

    fig, ax = plt.subplots(figsize=(9.5, 6.0))
    fit_rows = []

    for st in sorted(thr_df["studied"].unique()):
        for ch in sorted(thr_df.loc[thr_df["studied"] == st, "channel"].unique()):
            sel = thr_df[(thr_df["studied"] == st) & (thr_df["channel"] == ch)].sort_values("hv_V")
            for k, thr in enumerate(SIGNAL_THRESHOLDS_MV):
                _col = f"n_{abs(thr):.0f}mV"
                _ls = _thr_styles[k % len(_thr_styles)]
                _cc = COLORS.get(st, "tab:blue")
                x = sel["hv_V"].to_numpy(dtype=float)
                y = sel[_col].to_numpy(dtype=float)

                # dati: solo punti
                ax.errorbar(x, y, yerr=np.sqrt(np.maximum(y, 0)),
                            color=_cc, linestyle="none",
                            marker=_thr_marks[k % len(_thr_marks)], markersize=5,
                            capsize=2.5, alpha=0.9,
                            label=f"{st} (ch{ch}) \u2014 thr {thr:.0f} mV")

                # fit sigmoide -> curva
                A = v50 = kk = np.nan
                if FIT_COUNTS_SIGMOID and len(x) >= 4 and np.ptp(y) > 0:
                    _ymax = float(np.max(y))
                    try:
                        popt, _ = curve_fit(
                            _sigmoid, x, y,
                            p0=[_ymax, 1000.0, 40.0],
                            sigma=np.sqrt(np.maximum(y, 1)), absolute_sigma=True,
                            bounds=([0.5 * _ymax, 600.0, 3.0],
                                    [3.0 * _ymax + 5, 1800.0, 500.0]),
                            maxfev=20000,
                        )
                        A, v50, kk = (float(z) for z in popt)
                        xf = np.linspace(x.min(), x.max(), 300)
                        ax.plot(xf, _sigmoid(xf, A, v50, kk),
                                color=_cc, linestyle=_ls, linewidth=1.4, alpha=0.9)
                    except Exception:
                        pass
                fit_rows.append({"studied": st, "channel": ch, "thr_mV": thr,
                                 "plateau": A, "V50_V": v50, "width_V": kk})

    ax.set_xlabel("HV of PMT under test [V]")
    ax.set_ylabel(f"Signal counts, dark rejected (out of {_ntrig} triggers/run)")
    ax.set_title(
        f"Dark-rejected counts vs HV at different amplitude thresholds\n"
        f"(gate {COUNT_WINDOW_NS[0]:.0f}\u2013{COUNT_WINDOW_NS[1]:.0f} ns, Q > median/{SIGNAL_Q_CUT_FACTOR:g}; "
        f"color = PMT, linestyle = threshold)",
        fontsize=10,
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7.5, ncol=2)
    plt.tight_layout()
    plt.show()

    if FIT_COUNTS_SIGMOID:
        fit_counts_df = pd.DataFrame(fit_rows)
        print("\nParametri sigmoide  n(HV) = A / (1 + exp(-(HV - V50)/k)):")
        display(fit_counts_df)
        fit_counts_df.to_csv("counts_sigmoid_fit.csv", index=False)

    display(thr_df)
    thr_df.to_csv("counts_vs_threshold.csv", index=False)
    print("Salvati conteggi vs soglia: counts_vs_threshold.csv")


# ============================================================
# CARICA DEI SEGNALI — fit Landau-like e MPV vs HV (guadagno)
# ============================================================
# La carica per evento (integral_mV_ns) e' gia' calcolata nel loop sul segnale
# con modo comune sottratto. Qui si fitta la distribuzione per ogni run e si
# studia l'MPV in funzione della HV, sovrapponendo le due configurazioni.

if PLOT_CHARGE and DEBUG_PLOT_CHARGE_HIST:
    _studs = sorted(df["studied"].unique())
    _hvs = sorted(df["hv_V"].dropna().unique())

    # --- selezioni, calcolate una volta sola ---
    _sig_mask = df["passed_count"] & df["t_min_ns"].between(*SIGNAL_TIME_WINDOW_NS)

    def _sig_of(st, hv):
        a = df.loc[(df["studied"] == st) & (df["hv_V"] == hv) & _sig_mask,
                   "integral_mV_ns"].to_numpy(dtype=float)
        return a[np.isfinite(a) & (a > 0)]

    def _ped_of(st, hv):
        a = df.loc[(df["studied"] == st) & (df["hv_V"] == hv), "ped_mV_ns"].to_numpy(dtype=float)
        return a[np.isfinite(a)]

    # --- RANGE GLOBALE: gli stessi bin per tutti i pannelli ---
    _all_sig = df.loc[_sig_mask, "integral_mV_ns"].to_numpy(dtype=float)
    _all_sig = _all_sig[np.isfinite(_all_sig) & (_all_sig > 0)]
    _all_ped = df["ped_mV_ns"].to_numpy(dtype=float)
    _all_ped = _all_ped[np.isfinite(_all_ped)]
    _ped_pos = _all_ped[_all_ped > 0]

    if not _studs or not _hvs or len(_all_sig) == 0:
        print("Nessun dato per gli istogrammi di carica.")
    else:
        if CHARGE_HIST_RANGE is not None:
            _lo, _hi = CHARGE_HIST_RANGE
        elif CHARGE_HIST_LOGX:
            # in log si vede solo la meta' positiva del piedistallo (centrato su 0)
            _lo = float(np.percentile(_ped_pos, 1.0)) if len(_ped_pos) else 1.0
            _hi = float(np.percentile(_all_sig, 99.9)) * 10 ** CHARGE_HIST_XMARGIN_DEC
        else:
            _lo = float(np.percentile(_all_ped, 0.5)) if len(_all_ped) else 0.0
            _hi = float(np.percentile(_all_sig, 99.9)) * 10 ** CHARGE_HIST_XMARGIN_DEC

        if CHARGE_HIST_LOGX and _lo > 0:
            _bins = np.logspace(np.log10(_lo), np.log10(_hi), CHARGE_HIST_BINS + 1)
        else:
            _bins = np.linspace(_lo, _hi, CHARGE_HIST_BINS + 1)

        _frac_neg = float((_all_ped <= 0).mean()) if len(_all_ped) else 0.0

        fig, axes = plt.subplots(
            len(_hvs), len(_studs),
            figsize=(5.5 * len(_studs), 2.1 * len(_hvs)),
            sharex=True, squeeze=False,
        )

        for r, hv in enumerate(_hvs):
            for c, st in enumerate(_studs):
                ax = axes[r][c]
                sig, ped = _sig_of(st, hv), _ped_of(st, hv)

                if len(ped):
                    _pw, _lbl = None, f"pedestal, n={len(ped)}"
                    if CHARGE_HIST_PED_SCALE and len(sig):
                        _hp, _ = np.histogram(ped, bins=_bins)
                        _hs, _ = np.histogram(sig, bins=_bins)
                        if _hp.max() > 0 and _hs.max() > 0:
                            _pw = np.full(len(ped), _hs.max() / _hp.max())
                            _lbl += " [rescaled]"
                    ax.hist(ped, bins=_bins, weights=_pw, histtype="stepfilled",
                            alpha=0.45, color="0.55", label=_lbl)

                if len(sig):
                    ax.hist(sig, bins=_bins, histtype="step", lw=1.5,
                            color=COLORS.get(st, "tab:blue"),
                            label=f"in-time signal, n={len(sig)}")

                if CHARGE_HIST_LOGX and _lo > 0:
                    ax.set_xscale("log")
                if CHARGE_HIST_LOGY:
                    ax.set_yscale("log")
                ax.set_xlim(_lo, _hi)          # stessa scala X su tutti i pannelli
                ax.grid(True, alpha=0.25, which="both")
                ax.legend(fontsize=6.5, loc="upper left")
                ax.text(0.98, 0.92, f"{st} @ {hv:.0f} V", transform=ax.transAxes,
                        fontsize=8, va="top", ha="right",
                        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.8))
                if r == 0:
                    ax.set_title(f"PMT {st}")
                if c == 0:
                    ax.set_ylabel("Events", fontsize=8)

        for c in range(len(_studs)):
            axes[-1][c].set_xlabel("Integrated charge [mV\u00b7ns]")

        _note = (f"  \u2014 log scale hides the negative half of the pedestal "
                 f"({_frac_neg:.0%} of events)") if CHARGE_HIST_LOGX else ""
        fig.suptitle(
            f"Charge distributions \u2014 signal (gate "
            f"{INTEGRATION_WINDOW_NS[0]:.0f}\u2013{INTEGRATION_WINDOW_NS[1]:.0f} ns) vs pedestal "
            f"({PEDESTAL_WINDOW_NS[0]:.0f}\u2013{PEDESTAL_WINDOW_NS[1]:.0f} ns){_note}",
            y=1.0, fontsize=11,
        )
        plt.tight_layout()
        plt.show()


if PLOT_CHARGE:
    mpv_results = []

    for (studied_r, run_r), run_df in df.groupby(["studied", "run"]):
        hv_r      = run_df["hv_V"].iloc[0]
        hv_trig_r = run_df["hv_trig_V"].iloc[0]

        for ch in sorted(run_df["channel"].unique()):
            # Solo eventi IN TEMPO col trigger -> esclude i dark count
            _m = ((run_df["channel"] == ch) & run_df["passed_count"]
                  & run_df["t_min_ns"].between(*SIGNAL_TIME_WINDOW_NS))
            vals = run_df.loc[_m, "integral_mV_ns"].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals) & (vals > 0)]

            # Range di fit adattivo: il picco si sposta di decadi con l'HV
            if FIT_RANGE_MODE == "auto" and len(vals) >= 20:
                _fr = (float(np.percentile(vals, FIT_RANGE_PCT[0])),
                       float(np.percentile(vals, FIT_RANGE_PCT[1])))
            else:
                _fr = FIT_RANGE

            if MPV_METHOD == "median":
                fitres = estimate_median_charge(vals)
            elif MPV_METHOD == "logpeak":
                fitres = estimate_mpv_log(vals)
            else:
                fitres = fit_landau_histogram(vals, nbins=HIST_BINS, fit_range=_fr)

            mpv_results.append({
                "studied": studied_r, "run": run_r, "hv_V": hv_r,
                "hv_trig_V": hv_trig_r, "channel": ch,
                "mpv_mV_ns":     fitres["mpv"]     if fitres else np.nan,
                "mpv_err_mV_ns": fitres["mpv_err"] if fitres else np.nan,
                "n_entries_fit": len(vals),
            })

            if DEBUG_PLOT_CHARGE_HIST_SINGLE:
                fig, ax = plt.subplots(figsize=(9.0, 3.6))
                ax.hist(vals, bins=HIST_BINS, range=_fr, histtype="stepfilled",
                        alpha=0.45, color="skyblue",
                        label=f"PMT {studied_r} (ch{ch}) — {len(vals)} entries")
                if fitres is not None and "params" in fitres:
                    _x = np.linspace(_fr[0], _fr[1], 1200)
                    ax.plot(_x, landau_like_pdf(_x, *fitres["params"]), color="red", lw=2.0,
                            label=f"Landau-like | MPV = {fitres['mpv']:.1f} \u00b1 {fitres['mpv_err']:.1f}")
                ax.set_xlim(*_fr)
                ax.set_xlabel("Integrated charge [mV\u00b7ns]")
                ax.set_ylabel("Events")
                ax.set_title(f"PMT {studied_r} @ {hv_r} V (trig {hv_trig_r} V) | channel {ch}", fontsize=10)
                ax.grid(True, alpha=0.25)
                ax.legend(fontsize=9)
                plt.tight_layout()
                plt.show()

    df_mpv = pd.DataFrame(mpv_results).sort_values(["studied", "channel", "hv_V"])
    display(df_mpv)
    df_mpv.to_csv(MPV_CSV, index=False)
    print(f"Salvato file MPV: {MPV_CSV}")

    # --- MPV vs HV: le due configurazioni sovrapposte ---
    _v = df_mpv.dropna(subset=["hv_V", "mpv_mV_ns"])
    _nfit = int((~df_mpv["mpv_mV_ns"].notna()).sum())
    if _nfit:
        print(f"[ATTENZIONE] fit non riuscito per {_nfit} run -> punti assenti dal plot MPV")

    if len(_v) == 0:
        print("Nessun punto valido per il plot MPV vs HV.")
    else:
        fig, ax = plt.subplots(figsize=(9.0, 6.0))
        for st in sorted(_v["studied"].unique()):
            for ch in sorted(_v.loc[_v["studied"] == st, "channel"].unique()):
                sel = _v[(_v["studied"] == st) & (_v["channel"] == ch)].sort_values("hv_V")
                ax.errorbar(
                    sel["hv_V"], sel["mpv_mV_ns"], yerr=sel["mpv_err_mV_ns"],
                    marker=MARKERS.get(st, "o"), linestyle="-", linewidth=1.6, capsize=3,
                    color=COLORS.get(st), label=f"PMT {st} (ch{ch})",
                )
        ax.set_xlabel("HV of PMT under test [V]")
        _lab = "Median charge" if MPV_METHOD == "median" else "MPV"
        ax.set_ylabel(f"{_lab} [mV\u00b7ns]")
        ax.set_yscale("log")   # il guadagno del PMT e' esponenziale in HV
        ax.set_title(
            f"{_lab} vs HV \u2014 in-time events only "
            f"({SIGNAL_TIME_WINDOW_NS[0]:.0f}\u2013{SIGNAL_TIME_WINDOW_NS[1]:.0f} ns, dark counts rejected)",
            fontsize=11,
        )
        ax.grid(True, alpha=0.3, which="both")
        ax.legend()
        plt.tight_layout()
        plt.show()
